# 04 · Knowledge Distillation — Standard + Reliability-Aware

Run only after:

1. Text teacher notebook is locked.
2. Audio teacher notebook is locked.
3. No-KD student baseline is working.

This notebook is the only student-training notebook allowed to consume teacher probabilities/logits.

KD protocol will broadcast each participant's locked teacher target to that participant's student segments, while checkpoint selection/evaluation remains participant-level. TEST stays closed.


In [ ]:
from google.colab import drive,userdata
drive.mount('/content/drive')

from pathlib import Path
import base64,subprocess,sys,json
REPO='engrkhubabahmad/reliability-aware-depression-kd'
ROOT=Path('/content/reliability-aware-depression-kd')
token=userdata.get('GITHUB_TOKEN'); assert token,'Add GITHUB_TOKEN to Colab Secrets'
auth=base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git=['git','-c',f'http.extraHeader=AUTHORIZATION: basic {auth}']
if not (ROOT/'.git').exists():
    subprocess.run(git+['clone',f'https://github.com/{REPO}.git',str(ROOT)],check=True)
subprocess.run(git+['-C',str(ROOT),'fetch','origin','main'],check=True)
subprocess.run(['git','-C',str(ROOT),'checkout','main'],check=True)
subprocess.run(git+['-C',str(ROOT),'pull','--ff-only','origin','main'],check=True)
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'requirements-colab.txt')])
print('main:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())


## Verify locked teacher artifacts


In [ ]:
import pandas as pd,numpy as np
A=Path('/content/drive/MyDrive/DAIC_WOZ/experiments/ussd_audio_teacher_final/train_run4_crop_kd_targets.csv')
AD=Path('/content/drive/MyDrive/DAIC_WOZ/experiments/ussd_frozen_audit/dev_participant_predictions.csv')
T=Path('/content/drive/MyDrive/DAIC_WOZ/experiments/idiap_text_teacher/train_text_kd_targets.csv')
TD=Path('/content/drive/MyDrive/DAIC_WOZ/experiments/idiap_text_teacher/dev_text_predictions.csv')
for p in (A,AD,T,TD): assert p.exists(),p
a,t=pd.read_csv(A),pd.read_csv(T); ad,td=pd.read_csv(AD),pd.read_csv(TD)
assert len(a)==len(t)==107 and set(a.participant_id)==set(t.participant_id)
assert len(ad)==len(td)==34 and set(ad.participant_id)==set(td.participant_id)
assert 440 not in set(ad.participant_id)
print('Locked teacher alignment preflight: OK')


## Reliability rule

Do **not** use raw entropy confidence across the two teachers because their logit scales differ strongly.

The active KD implementation must estimate a scale per teacher from TRAIN outputs only, then compute scale-normalized confidence. No DEV labels and no TEST data may determine teacher reliability.


In [ ]:
az=np.median(np.abs(a.kd_logit.to_numpy(float)))
tz=np.median(np.abs(t.text_logit.to_numpy(float)))
print('TRAIN median |audio logit|:',az)
print('TRAIN median |text logit| :',tz)
assert az>0 and tz>0
print('Reliability scales available from TRAIN only.')


## KD runner gate

The participant-correct KD implementation will live under `scripts/kd/participant_kd/`. This prevents accidental execution of the obsolete segment-level KD code.


In [ ]:
RUNNER=ROOT/'scripts/kd/participant_kd/train_kd.py'
assert RUNNER.exists(), (
 'Participant-correct KD runner is not committed yet. '
 'Do not substitute scripts/students/train_segment_students.py or the old entropy target builder.'
)
subprocess.run([sys.executable,str(RUNNER)],cwd=ROOT,check=True)
